<a href="https://colab.research.google.com/github/aditi-govindu/gemini-recipe-agent/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [47]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('AQ.Ab8RN6K63OP5lQRCNCaFZi-ThKwWn64NLje-nlbh7FqJ81trpg')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [48]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [49]:
# @title User input needed.


# @title User input needed

DOCUMENTS = {
    "pasta": (
        "Pasta is a popular Italian food made from wheat flour and water. "
        "Boil pasta in water until it becomes soft. "
        "Add tomato sauce, vegetables, salt, and pepper for flavour. "
        "Serve pasta hot after mixing it with the sauce."
    ),

    "pizza": (
        "Pizza is an Italian dish made with a flat bread base. "
        "Spread tomato sauce over the base and add cheese. "
        "Add vegetables such as onions, capsicum, and tomatoes. "
        "Bake the pizza until the cheese melts and the crust becomes golden."
    ),

    "rice": (
        "Rice is a staple food eaten in many countries. "
        "Wash rice with clean water before cooking. "
        "Add the required amount of water and cook until the rice is soft. "
        "Rice can be served with vegetables, curry, or dal."
    ),

    "food_safety": (
        "Wash your hands before preparing food. "
        "Use clean utensils and fresh ingredients while cooking. "
        "Store leftover food in a refrigerator to keep it fresh. "
        "Cook food properly before serving it."
    )
}


In [50]:
# @title Define a tool
def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""
    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(1 for word in words if word in content.lower())
        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []
    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")
    return "\n\n".join(output)


# Sanity check for regex match.
print(search_documents("pasta"))


[Source: pasta]
Pasta is a popular Italian food made from wheat flour and water. Boil pasta in water until it becomes soft. Add tomato sauce, vegetables, salt, and pepper for flavour. Serve pasta hot after mixing it with the sauce.


In [51]:

# @title Ask Gemini to search for something

MODEL_NAME = "gemini-3-flash-preview"

SYSTEM_INSTRUCTION = """
You are a chef training at a Michelin star restaurant.
Use the search_documents tool when you need information
about cooking, ingredients, or food safety.
Only answer based on information found in the documents.
If the documents do not contain the answer, say so clearly.
"""

print("Model ready:", MODEL_NAME)


Model ready: gemini-3-flash-preview


In [52]:
import time
from google import genai
from google.genai import types

client = genai.Client()  # reads GEMINI_API_KEY / GOOGLE_API_KEY from env

search_tool = types.Tool(function_declarations=[
    types.FunctionDeclaration(
        name="search_documents",
        description="Search the documents to find most the relevant paragraph.",
        parameters={
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    )
])

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)


def send_with_retry(contents, max_retries=3):
    for attempt in range(max_retries):
        try:
            return client.models.generate_content(
                model=MODEL_NAME,
                contents=contents, config=config
            )
        except Exception as e:
            if "429" in str(e) and attempt < max_retries - 1:
                wait = 20 * (attempt + 1)
                print(f"  Rate limit hit, waiting {wait}s...")
                time.sleep(wait)
            else:
                raise


def run_agent(question: str, max_steps: int = 8) -> str:
    print(f"Question: {question}\n" + "-" * 50)
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    for step in range(1, max_steps + 1):
        response = send_with_retry(contents)
        model_content = response.candidates[0].content

        # Append the model's turn UNMODIFIED, so thought signatures are preserved
        contents.append(model_content)

        calls = [p.function_call for p in model_content.parts if p.function_call]
        if not calls:
            print(f"\nFinal answer:\n{response.text}")
            return response.text

        # Run every requested tool call, reply with all results in ONE user turn
        result_parts = []
        for call in calls:
            args = dict(call.args)
            print(f"Step {step}: model calls {call.name}({args})")
            result = search_documents(**args)
            print(f"  Result preview: {str(result)[:100]}...")
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )
        contents.append(types.Content(role="user", parts=result_parts))

    return "(Stopped: hit max_steps)"


_ = run_agent("What topics are students supposed to share documents on?")

Question: What topics are students supposed to share documents on?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'topics students share documents on'})
  Result preview: [Source: pizza]
Pizza is an Italian dish made with a flat bread base. Spread tomato sauce over the b...
Step 2: model calls search_documents({'query': 'student document sharing topics'})
  Result preview: No relevant documents found for that query....
Step 3: model calls search_documents({'query': 'student'})
  Result preview: No relevant documents found for that query....
Step 4: model calls search_documents({'query': 'topics'})
  Result preview: No relevant documents found for that query....
Step 5: model calls search_documents({'query': 'share document'})
  Result preview: No relevant documents found for that query....

Final answer:
I'm sorry, but the provided documents do not contain information regarding topics that students are supposed to share documents on. T

In [53]:
# @title Ask follow up questions

# TODO: Enter questions based on documents for the agent to answer.
questions = [
    "What ingredients can be added to pasta?",
    "How should pizza be prepared and baked?",
    "What steps should be followed for food safety?",
    "What is the most popular food item in Canada?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # stay within the free-tier rate limit.
    print("\n" + "=" * 60)
    try:
        run_agent(q)
    except Exception as e:
        print(f"An error occurred while running the agent: {e}")
        print("Skipping to the next question in 10 seconds...")
        time.sleep(10)


Question: What ingredients can be added to pasta?
--------------------------------------------------
Step 1: model calls search_documents({'query': 'ingredients to add to pasta list proteins vegetables sauces cheeses herbs'})
  Result preview: [Source: pasta]
Pasta is a popular Italian food made from wheat flour and water. Boil pasta in water...
  Rate limit hit, waiting 20s...
Step 2: model calls search_documents({'query': 'diverse ingredients for pasta proteins cheeses vegetables nuts oils herbs seafood'})
  Result preview: [Source: pasta]
Pasta is a popular Italian food made from wheat flour and water. Boil pasta in water...
  Rate limit hit, waiting 20s...
  Rate limit hit, waiting 40s...
Step 3: model calls search_documents({'query': 'pasta toppings add-ins list recipes protein vegetable cheese herb sauce'})
  Result preview: [Source: pizza]
Pizza is an Italian dish made with a flat bread base. Spread tomato sauce over the b...

Final answer:
Pasta is incredibly versatile, and yo

In [54]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)